In [1]:
from pathlib import Path

import pandas as pd


def load_folder(folder):
    files = sorted(Path(folder).glob("*.parquet"))
    return pd.concat((pd.read_parquet(f) for f in files), ignore_index=True)

In [2]:
from pathlib import Path

def load_folder(folder):
    files = sorted(Path(folder).glob("*.parquet"))
    return pd.concat((pd.read_parquet(f) for f in files), ignore_index=True)

neso = load_folder("../data/raw/neso_intensity")
neso["year"] = neso["period_start"].dt.year
print(neso.groupby("year").agg(
    rows=("period_start", "size"),
    null_actual=("actual", lambda s: s.isna().sum()),
    null_forecast=("forecast", lambda s: s.isna().sum()),
))

       rows  null_actual  null_forecast
year                                   
2023  17429            5              0
2024  17537            0              0
2025  17520            0             43
2026  13848            0              0


In [3]:
weather = load_folder("../data/raw/weather")
weather["year"] = weather["time"].dt.year
weather.groupby(["location", "year"]).agg(
    rows=("time", "size"),
    null_wind=("wind_speed_100m", lambda s: s.isna().sum()),
    null_solar=("shortwave_radiation", lambda s: s.isna().sum()),
)

rows  null_wind  null_solar
location            year                             
london              2023  8760          0           0
                    2024  8784          0           0
                    2025  8760          0           0
                    2026  6792          0           0
north_sea           2023  8760          0           0
                    2024  8784          0           0
                    2025  8760          0           0
                    2026  6792          0           0
scotland_wind       2023  8760          0           0
                    2024  8784          0           0
                    2025  8760          0           0
                    2026  6792          0           0
south_england_solar 2023  8760          0           0
                    2024  8784          0           0
                    2025  8760          0           0
                    2026  6792          0           0

In [4]:
expected = pd.date_range(neso["period_start"].min(), neso["period_start"].max(), freq="30min")
missing = expected.difference(neso["period_start"])
print("Missing periods:", len(missing))

# Group consecutive missing periods into gaps
gaps = (
    pd.Series(missing)
    .groupby((pd.Series(missing).diff() != pd.Timedelta("30min")).cumsum())
    .agg(["min", "max", "size"])
    .rename(columns={"min": "gap_start", "max": "gap_end", "size": "periods"})
)
gaps


Missing periods: 122


,gap_start,gap_end,periods
1,2023-10-20 22:00:00+00:00,2023-10-22 19:00:00+00:00,91
2,2024-06-11 23:00:00+00:00,2024-06-12 14:00:00+00:00,31


In [5]:
import requests

for s, e in [("2023-10-20T21:30Z", "2023-10-22T19:30Z"),
             ("2024-06-11T22:30Z", "2024-06-12T14:30Z")]:
    r = requests.get(f"https://api.carbonintensity.org.uk/intensity/{s}/{e}",
                     headers={"Accept": "application/json"}, timeout=30)
    rows = r.json().get("data", [])
    print(f"{s} to {e}: status {r.status_code}, rows returned: {len(rows)}")

2023-10-20T21:30Z to 2023-10-22T19:30Z: status 200, rows returned: 2
2024-06-11T22:30Z to 2024-06-12T14:30Z: status 200, rows returned: 2


## Data quality findings

**Weather (Open-Meteo archive):** 4 locations, Jan 2023 to Oct 2026. Full hourly coverage, zero nulls.

**NESO national intensity:** 66,334 of 66,456 expected half-hours (99.8%).
Two source outages, confirmed by direct re-fetch (API returns only boundary periods):
- 2023-10-20 22:00 to 2023-10-22 19:00 UTC: 91 periods (~45h)
- 2024-06-11 23:00 to 2024-06-12 14:00 UTC: 31 periods (~15.5h)

**Decision:** gaps are too long to interpolate safely (the 2023 gap spans almost two full daily cycles). Raw data is left untouched; in feature engineering, rows whose target or lag features fall inside a gap are excluded from training.

**Nulls within available rows:**
- `actual`: 5 nulls (2023). Excluded as training targets.
- `forecast`: 43 nulls (2025). Excluded from the NESO benchmark comparison only; not used as a model feature.

All row counts reconcile with expected half-hours minus the two outages.